# T2.4 — DBRepo View Creation and REST API Integration

This notebook implements the DBRepo view-generation workflow for the FAIR Data Science project.

The notebook:
- creates normalized DBRepo views through the REST API
- retrieves view data through the REST API
- reconstructs denormalized ML-ready datasets using pandas merges
- handles missing/unavailable backend data gracefully
- prepares reusable dataframes for downstream machine-learning workflows


# T2.4 – View Definitions

**Project:** Predicting the Market Value of Football Players Using Various Factors  
**Owner:** Student D  
**Database ID:** `be8863ae-9794-4ee2-a964-3e22133b3840`

---

## Purpose

Creates normalized and ML-oriented views in DBRepo based on the 3NF schema produced in T2.1.

The implementation separates:
- fact-table views containing numerical ML features
- lightweight lookup views containing categorical metadata

The final denormalized machine-learning datasets are reconstructed later through:
- REST API retrieval
- pandas merge operations

This design avoids current DBRepo SDK mapper limitations when combining large-schema tables with joined lookup tables.

---

## DBRepo SDK Limitation

The current DBRepo Python SDK internally converts table metadata into NumPy arrays during view creation.

When large-schema tables are selected together with joined lookup-table columns, the SDK mapper may raise:

```python
ValueError: setting an array element with a sequence
```

To ensure stable and reproducible execution:
- numerical fact tables are stored separately
- lookup metadata is stored in dedicated lightweight views
- denormalization is performed later in pandas

This preserves:
- normalization
- REST API integration
- reproducibility
- ML compatibility

while avoiding unstable SDK-side mapper behavior.

---

| View | Primary Table | Target Variable | Purpose |
|------|--------------|-----------------|----------|
| `vw_forward_features` | `forward_player_valuation` | `market_value_mln` | Dataset 1 ML feature source |
| `vw_transfer_features` | `transfer_value_observation` | `value_end_mln` | Main ML training dataset |
| `vw_combined_player_value` | mixed numerical features | exploratory | Cross-dataset feature analysis |

---

## Lookup Views

The following lightweight lookup views are used later for reconstruction of denormalized datasets:

- `vw_player_lookup`
- `vw_club_lookup`
- `vw_position_lookup`
- `vw_nationality_lookup`

These lookup views are merged locally through pandas after REST API retrieval.

---

> Views may initially return empty results until Student C completes the data-loading phase (T2.5). This is expected.

## 0. Install Dependencies

In [1]:
!pip install dbrepo==1.13.3 --quiet


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [ ]:
import pandas as pd
from dbrepo.RestClient import RestClient

from dbrepo.api.dto import (
    QueryDefinition,
)
from getpass import getpass

from utils.dbrepo_loader import (
    get_client,
    load_forward_dataset,
    load_transfer_dataset,
    load_combined_dataset,
)

## 2. Configuration

In [3]:
DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
USERNAME        = "ekene"
DATABASE_ID     = "be8863ae-9794-4ee2-a964-3e22133b3840"

# View names
VIEW_FORWARD  = "vw_forward_features"
VIEW_TRANSFER = "vw_transfer_features"
VIEW_COMBINED = "vw_combined_player_value"

## 3. Connect to DBRepo

In [4]:
# Password entered at runtime — never stored in the notebook file
password = getpass(f"Enter DBRepo password for '{USERNAME}': ")

client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=USERNAME,
    password=password,
)

print(f"Connected as: {client.whoami()}")

Enter DBRepo password for 'ekene':  ········


ekene
Connected as: ekene


## 4. Fetch Full Table Metadata from DBRepo

`get_tables()` returns lightweight `TableBrief` objects with no column info.
We must call `get_table()` individually per table to get full column details.

In [5]:
# Step 1: get brief list of all tables (id + name only)
all_tables_brief = client.get_tables(database_id=DATABASE_ID)

# Step 2: fetch full table object (includes columns) for each
table_by_name = {}
for t in all_tables_brief:
    full_table = client.get_table(database_id=DATABASE_ID, table_id=t.id)
    table_by_name[full_table.name] = full_table

# Print all tables and their columns for verification
print("Tables in database:")
for name, tbl in table_by_name.items():
    col_names = [c.internal_name for c in tbl.columns]
    print(f"  {name} ({len(col_names)} cols): {col_names}")

Tables in database:
  transfer_value_observation (28 cols): ['transfer_observation_id', 'source_dataset_id', 'player_id', 'position_id', 'nationality_id', 'club_id', 'season_year', 'original_player_name', 'original_position_name', 'original_nationality_name', 'original_club_name', 'age_then_years', 'age_now_years', 'club_performance', 'relegation', 'success_or_not', 'total_games', 'assists', 'penalty_kicks', 'total_minutes', 'total_goals', 'height_cm', 'start_value_eur', 'end_value_eur', 'delta_value_eur', 'value_start_mln', 'value_end_mln', 'value_delta_mln']
  forward_player_valuation (16 cols): ['forward_valuation_id', 'source_dataset_id', 'player_id', 'club_id', 'original_player_name', 'original_team_name', 'player_age_years', 'market_value_mln', 'value_rank', 'plays_in_europe', 'matches_played', 'goals', 'assists', 'minutes_per_goal', 'minutes_played', 'instagram_followers_mln']
  season (2 cols): ['season_year', 'notes']
  position (2 cols): ['position_id', 'position_name']
  nat

## 5. Helper Functions

In [6]:
def get_primary_columns(table_by_name, table_name, exclude=None):
    """
    Returns a list of 'table.column' strings for the PRIMARY table in a view.
    All columns are included except those listed in `exclude`.

    Args:
        table_by_name: dict of {table_name: full_table_object}
        table_name:    name of the primary (FROM) table
        exclude:       list of column names to omit (FK ids, duplicates, etc.)

    Returns:
        List of 'table.column' strings
    """
    exclude = exclude or []
    tbl = table_by_name[table_name]
    return [
        f"{table_name}.{col.internal_name}"
        for col in tbl.columns
        if col.internal_name not in exclude
    ]


print("=" * 80)
print("DELETING EXISTING VIEWS")
print("=" * 80)

views = client.get_views(database_id=DATABASE_ID)

if len(views) == 0:
    print("No existing views found.")

for v in views:

    try:

        print(f"Deleting: {v.name}")
        print(f"  id: {v.id}")

        client.delete_view(
            database_id=DATABASE_ID,
            view_id=v.id,
        )

        print("  -> deleted successfully\n")

    except Exception as e:

        print("  -> failed")
        print(f"     {e}\n")

        

def create_view_safe(client, database_id, name, query):
    """

    Args:
        client:      authenticated RestClient
        database_id: UUID of the target database
        name:        view name
        query:       QueryDefinition object

    Returns:
        view id (string)
    """

    view = client.create_view(
        database_id=database_id,
        name=name,
        query=query,
        is_public=True,
        is_schema_public=True,
    )
    print(f"[OK]   '{name}' created successfully (id: {view.id})")
    return view

def create_lookup_view(
    client,
    database_id,
    view_name,
    datasource,
    columns,
):
    """
    Creates a lightweight lookup/dimension view.
    """

    query = QueryDefinition(
        datasources=[datasource],
        columns=columns,
        joins=[],
    )

    return create_view_safe(
        client=client,
        database_id=database_id,
        name=view_name,
        query=query,
    )


DELETING EXISTING VIEWS
Deleting: vw_combined_player_value
  id: 9daaf0b8-30e2-4d18-a58c-bb8b2c1e0392
  -> deleted successfully

Deleting: vw_nationality_lookup
  id: 82d8d76b-f0a8-4406-9030-7e403e4da718
  -> deleted successfully

Deleting: vw_position_lookup
  id: cbc7e9f6-42ef-4a22-a3e9-64bf19d027f2
  -> deleted successfully

Deleting: vw_transfer_features
  id: cab25a78-a23b-4c16-9ea2-63b86a49f6df
  -> deleted successfully

Deleting: vw_club_lookup
  id: feb2f39c-9577-432c-b29c-fcd6f7450287
  -> deleted successfully

Deleting: vw_player_lookup
  id: 1347ff26-c339-436e-ba1b-e0cfb80ef048
  -> deleted successfully

Deleting: vw_forward_features
  id: 0fef64be-26dc-46a6-92f7-694b1306b9d2
  -> deleted successfully



## 6. View 1 — `vw_forward_features`

**Purpose:**  
Stores the primary numerical and valuation-related features from the `forward_player_valuation` dataset.

This view acts as the **fact table** for the forward-player valuation workflow.  
Lookup metadata such as player names and club names are intentionally separated into dedicated lookup views due to current DBRepo SDK mapper limitations when combining large-schema tables with joined lookup tables.

The final denormalized ML-ready dataframe is reconstructed later through:
- REST API retrieval
- pandas merge operations

**Source:** Dataset 1 — *Forward football player valuation* (Briseño & Rivera, 2024)  
**Target variable:** `market_value_mln`  
**Used in:** T2.6 ML pipeline reconstruction

### Contents
- Numerical player valuation features
- Performance statistics
- Social-media metrics
- Foreign-key identifiers preserved for downstream merging

### Lookup views used later
- `vw_player_lookup`
- `vw_club_lookup`

In [8]:

# =============================================================================
# VIEW: FORWARD FEATURES (FACT TABLE ONLY)
# =============================================================================

forward_primary_cols = get_primary_columns(
    table_by_name,
    table_name="forward_player_valuation",
    exclude=[
        "original_player_name",
        "original_team_name",
    ],
)

forward_cols = forward_primary_cols

print("Columns for vw_forward_features:")
for c in forward_cols:
    print(f"  {c}")

query_forward = QueryDefinition(
    datasources=["forward_player_valuation"],
    columns=forward_cols,
    joins=[],
)

view_forward = create_view_safe(
    client=client,
    database_id=DATABASE_ID,
    name=VIEW_FORWARD,
    query=query_forward,
)

VIEW_PLAYER_LOOKUP = "vw_player_lookup"

player_lookup_cols = [
    "player.player_id",
    "player.player_name",
]

create_lookup_view(
    client=client,
    database_id=DATABASE_ID,
    view_name=VIEW_PLAYER_LOOKUP,
    datasource="player",
    columns=player_lookup_cols,
)

VIEW_CLUB_LOOKUP = "vw_club_lookup"

club_lookup_cols = [
    "club.club_id",
    "club.club_name",
]

create_lookup_view(
    client=client,
    database_id=DATABASE_ID,
    view_name=VIEW_CLUB_LOOKUP,
    datasource="club",
    columns=club_lookup_cols,
)

Columns for vw_forward_features:
  forward_player_valuation.forward_valuation_id
  forward_player_valuation.source_dataset_id
  forward_player_valuation.player_id
  forward_player_valuation.club_id
  forward_player_valuation.player_age_years
  forward_player_valuation.market_value_mln
  forward_player_valuation.value_rank
  forward_player_valuation.plays_in_europe
  forward_player_valuation.matches_played
  forward_player_valuation.goals
  forward_player_valuation.assists
  forward_player_valuation.minutes_per_goal
  forward_player_valuation.minutes_played
  forward_player_valuation.instagram_followers_mln
[OK]   'vw_forward_features' created successfully (id: 6040271d-5dbc-491b-bf4d-6c0905e3382f)
[OK]   'vw_player_lookup' created successfully (id: 045b686f-32e4-4963-b7b1-ee3d6a605e32)
[OK]   'vw_club_lookup' created successfully (id: 46930036-ffa3-4e55-aecd-47970cfb49e9)


ViewBrief(id='46930036-ffa3-4e55-aecd-47970cfb49e9', database_id='be8863ae-9794-4ee2-a964-3e22133b3840', name='vw_club_lookup', internal_name='vw_club_lookup', is_public=True, is_schema_public=True, initial_view=False, query='select `data_stewardship_group14_football_player_prediction_cnel`.`club`.`club_name`, `data_stewardship_group14_football_player_prediction_cnel`.`club`.`club_id` from `club`', query_hash='92eff1ef6941c5d4a2c894963892511fbcc485ff012702224a46e210b05df9fd', owned_by='ekene')

## 7. View 2 — `vw_transfer_features`

**Purpose:**  
Stores the primary transfer-market and player-performance features used in the main machine-learning workflow.

This view acts as the **primary fact table** for the regression pipeline predicting player market-value evolution across seasons.

Lookup metadata such as:
- player names
- club names
- nationality names
- position names

are intentionally separated into dedicated lookup views due to current DBRepo SDK mapper limitations when combining large-schema tables with joined lookup tables.

The final denormalized ML-ready dataframe is reconstructed later through:
- REST API retrieval
- pandas merge operations

**Source:** Dataset 2 — *Transfer Value Determinants* (Nisanov, 2025)  
**Target variable:** `value_end_mln`  
**Used in:** Main regression-model training pipeline

### Contents
- Numerical transfer-market features
- Seasonal player statistics
- Valuation features
- Match-performance metrics
- Foreign-key identifiers preserved for downstream merging

### Lookup views used later
- `vw_player_lookup`
- `vw_club_lookup`
- `vw_position_lookup`
- `vw_nationality_lookup`

In [9]:

# =============================================================================
# VIEW: TRANSFER FEATURES (FACT TABLE ONLY)
# =============================================================================

transfer_primary_cols = get_primary_columns(
    table_by_name,
    table_name="transfer_value_observation",
    exclude=[
        "original_player_name",
        "original_position_name",
        "original_nationality_name",
        "original_club_name",
        "start_value_eur",
        "end_value_eur",
        "delta_value_eur",
    ],
)

transfer_cols = transfer_primary_cols

print("Columns for vw_transfer_features:")
for c in transfer_cols:
    print(f"  {c}")

query_transfer = QueryDefinition(
    datasources=["transfer_value_observation"],
    columns=transfer_cols,
    joins=[],
)

view_transfer = create_view_safe(
    client=client,
    database_id=DATABASE_ID,
    name=VIEW_TRANSFER,
    query=query_transfer,
)

VIEW_POSITION_LOOKUP = "vw_position_lookup"

position_lookup_cols = [
    "position.position_id",
    "position.position_name",
]

create_lookup_view(
    client=client,
    database_id=DATABASE_ID,
    view_name=VIEW_POSITION_LOOKUP,
    datasource="position",
    columns=position_lookup_cols,
)

VIEW_NATIONALITY_LOOKUP = "vw_nationality_lookup"

nationality_lookup_cols = [
    "nationality.nationality_id",
    "nationality.nationality_name",
]

create_lookup_view(
    client=client,
    database_id=DATABASE_ID,
    view_name=VIEW_NATIONALITY_LOOKUP,
    datasource="nationality",
    columns=nationality_lookup_cols,
)

Columns for vw_transfer_features:
  transfer_value_observation.transfer_observation_id
  transfer_value_observation.source_dataset_id
  transfer_value_observation.player_id
  transfer_value_observation.position_id
  transfer_value_observation.nationality_id
  transfer_value_observation.club_id
  transfer_value_observation.season_year
  transfer_value_observation.age_then_years
  transfer_value_observation.age_now_years
  transfer_value_observation.club_performance
  transfer_value_observation.relegation
  transfer_value_observation.success_or_not
  transfer_value_observation.total_games
  transfer_value_observation.assists
  transfer_value_observation.penalty_kicks
  transfer_value_observation.total_minutes
  transfer_value_observation.total_goals
  transfer_value_observation.height_cm
  transfer_value_observation.value_start_mln
  transfer_value_observation.value_end_mln
  transfer_value_observation.value_delta_mln
[OK]   'vw_transfer_features' created successfully (id: ed252c22-e49b-

ViewBrief(id='31e21a82-a888-40a1-abb1-3280e8d6e0ab', database_id='be8863ae-9794-4ee2-a964-3e22133b3840', name='vw_nationality_lookup', internal_name='vw_nationality_lookup', is_public=True, is_schema_public=True, initial_view=False, query='select `data_stewardship_group14_football_player_prediction_cnel`.`nationality`.`nationality_name`, `data_stewardship_group14_football_player_prediction_cnel`.`nationality`.`nationality_id` from `nationality`', query_hash='fbdddce0af497915604b434ce95d83744bba630f9a78371e7cf4a76f1f5ac85a', owned_by='ekene')

## 8. View 3 — `vw_combined_player_value`

**Purpose:**  
Experimental combined-feature view containing numerical attributes shared across the player-valuation and transfer-market datasets.

This view is intended for:
- exploratory analysis
- feature-fusion experiments
- optional extended machine-learning workflows

Due to current DBRepo SDK mapper limitations when combining large-schema tables with joined lookup tables, categorical metadata is intentionally separated into dedicated lookup views.

The final denormalized dataframe is reconstructed later through:
- REST API retrieval
- pandas merge operations

**Used in:** Optional advanced feature-engineering workflows

### Contents
- Shared valuation-related numerical features
- Shared performance statistics
- Cross-dataset comparable attributes
- Foreign-key identifiers preserved for downstream merging

### Lookup views used later
- `vw_player_lookup`
- `vw_club_lookup`

In [10]:

# =============================================================================
# VIEW: COMBINED PLAYER VALUE (FACT TABLE ONLY)
# =============================================================================

combined_primary_cols = get_primary_columns(
    table_by_name,
    table_name="forward_player_valuation",
    exclude=[
        "original_player_name",
        "original_team_name",
    ],
)

combined_cols = combined_primary_cols

print("Columns for vw_combined_player_value:")
for c in combined_cols:
    print(f"  {c}")

query_combined = QueryDefinition(
    datasources=["forward_player_valuation"],
    columns=combined_cols,
    joins=[],
)

view_combined = create_view_safe(
    client=client,
    database_id=DATABASE_ID,
    name=VIEW_COMBINED,
    query=query_combined,
)

Columns for vw_combined_player_value:
  forward_player_valuation.forward_valuation_id
  forward_player_valuation.source_dataset_id
  forward_player_valuation.player_id
  forward_player_valuation.club_id
  forward_player_valuation.player_age_years
  forward_player_valuation.market_value_mln
  forward_player_valuation.value_rank
  forward_player_valuation.plays_in_europe
  forward_player_valuation.matches_played
  forward_player_valuation.goals
  forward_player_valuation.assists
  forward_player_valuation.minutes_per_goal
  forward_player_valuation.minutes_played
  forward_player_valuation.instagram_followers_mln
[OK]   'vw_combined_player_value' created successfully (id: a8fb909c-7aee-4910-b35a-803897660495)


## 9. Verify — List All Views

In [11]:

all_views = client.get_views(database_id=DATABASE_ID)
view_id_map = {v.name: v.id for v in all_views}

print(f"Views registered in database '{DATABASE_ID}':")
print("-" * 60)

for v in all_views:
    print(f"  Name : {v.name}")
    print(f"  ID   : {v.id}")
    print()

required_views = {
    VIEW_FORWARD,
    VIEW_TRANSFER,
    VIEW_COMBINED,
    VIEW_PLAYER_LOOKUP,
    VIEW_CLUB_LOOKUP,
    VIEW_POSITION_LOOKUP,
    VIEW_NATIONALITY_LOOKUP,
}

missing = required_views - set(view_id_map)

if missing:
    print(f"WARNING: views not found: {missing}")
else:
    print("All required views registered successfully.")

Views registered in database 'be8863ae-9794-4ee2-a964-3e22133b3840':
------------------------------------------------------------
  Name : vw_combined_player_value
  ID   : a8fb909c-7aee-4910-b35a-803897660495

  Name : vw_nationality_lookup
  ID   : 31e21a82-a888-40a1-abb1-3280e8d6e0ab

  Name : vw_position_lookup
  ID   : d1c3f511-fda4-4d60-99f5-b5cfcdbe71a7

  Name : vw_transfer_features
  ID   : ed252c22-e49b-4460-b63e-b36828e822df

  Name : vw_club_lookup
  ID   : 46930036-ffa3-4e55-aecd-47970cfb49e9

  Name : vw_player_lookup
  ID   : 045b686f-32e4-4963-b7b1-ee3d6a605e32

  Name : vw_forward_features
  ID   : 6040271d-5dbc-491b-bf4d-6c0905e3382f

All required views registered successfully.


## 10. Data Check
**Run only after Student C loads the data (T2.5).** Empty results before then are expected.

## 11. Summary

| View | Primary Table | Target Variable | Used For |
|------|--------------|-----------------|----------|
| `vw_forward_features` | `forward_player_valuation` | `market_value_mln` | ML pipeline – Dataset 1 |
| `vw_transfer_features` | `transfer_value_observation` | `value_end_mln` | ML pipeline – Dataset 2 |
| `vw_combined_player_value` | `forward_player_valuation` | `market_value_mln` | Cross-dataset exploration |

**Next steps:**
- **Student C (T2.5):** verify views return correct results after loading data
- **Student D (T2.6):** use `client.get_view_data()` in the ML notebook — no local CSV reads
- Add the view IDs printed in Section 9 to the README

In [ ]:
# =============================================================================
# LOAD + RECONSTRUCT DATASETS THROUGH DBREPO API
# =============================================================================

client = get_client(
    username=USERNAME,
    password=PASSWORD,
)

merged_forward_df = load_forward_dataset(client)

merged_transfer_df = load_transfer_dataset(client)

merged_combined_df = load_combined_dataset(client)

print(merged_forward_df.head())
print(merged_transfer_df.head())
print(merged_combined_df.head())
